# Bootstrap Robustness Check — AA Smoking Status

**Purpose:** Assess how stable the PC algorithm's causal graph edges are to
resampling, as a complement to the alpha-sensitivity sweep (which varied the
significance threshold on fixed data; this varies the data itself via
bootstrap resampling with replacement).

**Method:** 100 bootstrap resamples of the 3,036-sample dataset, rerun PC
algorithm on each, count how often each gene appears as a direct parent of
smoking_status. Edges appearing in ≥70% of bootstraps are considered robust.

**Input:** `pc_input_smoking_final.npy`, `pc_col_names_smoking_final.json`
(the same 9-SNP final network from 04_pc_algorithm_smoking.ipynb).

In [1]:
import numpy as np
import json
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
from collections import defaultdict
import time

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
X_pc_full = np.load(os.path.join(out_dir, "pc_input_smoking_final.npy"))
with open(os.path.join(out_dir, "pc_col_names_smoking_final.json")) as f:
    col_names = json.load(f)

n_nodes = len(col_names)
outcome_idx = col_names.index("smoking_status")
n_samples = X_pc_full.shape[0]

def get_direct_parents(X, alpha_val=0.001):
    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in col_names]
    for i in range(n_nodes - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    cg = pc(data=X, alpha=alpha_val, indep_test=fisherz, stable=True,
            uc_rule=0, uc_priority=2, background_knowledge=bk,
            verbose=False, show_progress=False, node_names=col_names)

    adj = cg.G.graph
    direct_parents = set()
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if adj[i,j] == -1 and adj[j,i] == 1 and col_names[j] == "smoking_status":
                direct_parents.add(col_names[i])
            elif adj[i,j] == 1 and adj[j,i] == -1 and col_names[i] == "smoking_status":
                direct_parents.add(col_names[j])
            elif adj[i,j] == -1 and adj[j,i] == -1:
                if col_names[i] == "smoking_status":
                    direct_parents.add(col_names[j])
                elif col_names[j] == "smoking_status":
                    direct_parents.add(col_names[i])
    return direct_parents

n_bootstraps = 100
edge_counts = defaultdict(int)

np.random.seed(0)
start = time.time()
for b in range(n_bootstraps):
    boot_idx = np.random.choice(n_samples, n_samples, replace=True)
    X_boot = X_pc_full[boot_idx]
    parents = get_direct_parents(X_boot)
    for gene in parents:
        edge_counts[gene] += 1
    if (b + 1) % 20 == 0:
        print(f"Completed {b+1}/{n_bootstraps} bootstraps, elapsed {time.time()-start:.1f}s")

print(f"\nTotal time: {time.time()-start:.1f}s")
print("\nEdge stability across 100 bootstraps:")
for gene, count in sorted(edge_counts.items(), key=lambda x: -x[1]):
    print(f"  {gene}: {count}/100 ({count}%)")

robust_edges = {g: c for g, c in edge_counts.items() if c >= 70}
print(f"\nEdges appearing in >=70% of bootstraps: {len(robust_edges)}")
for g, c in sorted(robust_edges.items(), key=lambda x: -x[1]):
    print(f"  {g}: {c}%")

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Completed 20/100 bootstraps, elapsed 0.8s
Completed 40/100 bootstraps, elapsed 1.3s
Completed 60/100 bootstraps, elapsed 2.2s
Completed 80/100 bootstraps, elapsed 2.8s
Completed 100/100 bootstraps, elapsed 3.4s

Total time: 3.4s

Edge stability across 100 bootstraps:
  exm609218-0_B_F_1918575531: 75/100 (75%)
  exm782555-0_B_R_1922302526: 70/100 (70%)
  exm-rs7014346-131_B_R_1990484715: 69/100 (69%)
  exm1379120-0_T_F_1921625489: 63/100 (63%)
  exm100944-0_B_R_1921482882: 62/100 (62%)
  exm1245580-0_B_F_2060131617: 61/100 (61%)
  exm1003257-0_T_F_1922526121: 51/100 (51%)
  exm2277017-0_T_R_1989215336: 50/100 (50%)
  exm2941-0_B_R_1919117890: 45/100 (45%)

Edges appearing in >=70% of bootstraps: 2
  exm609218-0_B_F_1918575531: 75%
  exm782555-0_B_R_1922302526: 70%
